# Hyperparameter Tuning

Tune the top-performing models from the comparison step using `RandomizedSearchCV` with stratified 5-fold CV, then save the best pipeline.

> **Prerequisite**: Run notebooks 03 and 04 first.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from src.data import DATA_DIR
from src.preprocessing import build_preprocessor
from src.evaluate import plot_confusion_matrix, plot_roc_curve, print_report
from src.utils import save_model

import matplotlib.pyplot as plt
%matplotlib inline

## 1. Load Data

In [ ]:
df = pd.read_csv(DATA_DIR / "student-mat-engineered.csv")
X = df.drop(columns=["at_risk"])
y = df["at_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = build_preprocessor(X_train)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 2. Tune Random Forest

In [ ]:
rf_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=42)),
])

rf_params = {
    "clf__n_estimators": [100, 200, 300, 500],
    "clf__max_depth": [None, 5, 10, 15, 20],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
}

rf_search = RandomizedSearchCV(
    rf_pipe, rf_params, n_iter=40, scoring="f1", cv=skf, random_state=42, n_jobs=-1
)
rf_search.fit(X_train, y_train)

print(f"Best F1 (CV): {rf_search.best_score_:.4f}")
print(f"Best params: {rf_search.best_params_}")

## 3. Tune XGBoost

In [ ]:
xgb_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", XGBClassifier(
        use_label_encoder=False, eval_metric="logloss",
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        random_state=42, verbosity=0,
    )),
])

xgb_params = {
    "clf__n_estimators": [100, 200, 300],
    "clf__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "clf__max_depth": [3, 5, 7, 10],
    "clf__subsample": [0.7, 0.8, 0.9, 1.0],
    "clf__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
}

xgb_search = RandomizedSearchCV(
    xgb_pipe, xgb_params, n_iter=40, scoring="f1", cv=skf, random_state=42, n_jobs=-1
)
xgb_search.fit(X_train, y_train)

print(f"Best F1 (CV): {xgb_search.best_score_:.4f}")
print(f"Best params: {xgb_search.best_params_}")

## 4. Tune LightGBM

In [ ]:
lgbm_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1)),
])

lgbm_params = {
    "clf__n_estimators": [100, 200, 300],
    "clf__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "clf__max_depth": [-1, 5, 10, 15],
    "clf__subsample": [0.7, 0.8, 0.9, 1.0],
    "clf__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
}

lgbm_search = RandomizedSearchCV(
    lgbm_pipe, lgbm_params, n_iter=40, scoring="f1", cv=skf, random_state=42, n_jobs=-1
)
lgbm_search.fit(X_train, y_train)

print(f"Best F1 (CV): {lgbm_search.best_score_:.4f}")
print(f"Best params: {lgbm_search.best_params_}")

## 5. Compare Tuned Models & Select Best

In [ ]:
tuned = {
    "Random Forest": rf_search,
    "XGBoost": xgb_search,
    "LightGBM": lgbm_search,
}

for name, search in tuned.items():
    print(f"{name}: best CV F1 = {search.best_score_:.4f}")

best_name = max(tuned, key=lambda k: tuned[k].best_score_)
best_model = tuned[best_name].best_estimator_
print(f"\nBest model: {best_name}")

## 6. Test-Set Evaluation of Best Model

In [ ]:
y_pred = best_model.predict(X_test)

print(f"=== {best_name} (tuned) ===")
print_report(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_confusion_matrix(y_test, y_pred, title=f"{best_name} — Confusion Matrix", ax=axes[0])
plot_roc_curve(best_model, X_test, y_test, title=f"{best_name} — ROC Curve", ax=axes[1])
plt.tight_layout()
plt.show()

## 7. Save Best Model

In [ ]:
save_model(best_model, "best_model.joblib")